# Taiwan ASR Toolkit — Quickstart

Production-grade Traditional Chinese (Taiwan Mandarin) speech-to-text:
**MediaTek Breeze-ASR-25** + **Qwen3-ASR-1.7B** on an identical pipeline,
with hot-word injection, OpenCC `s2twp` post-processing, and 56 TDD tests.

**Repo**: https://github.com/thc1006/taiwan-asr-toolkit · MIT · 56 tests

## Pick a runtime first

Click `Runtime > Change runtime type` and pick a GPU.  This notebook works on
any GPU Colab gives you (or on CPU, slowly).  The toolkit auto-detects compute
capability **and VRAM tier**, then picks dtype + batch size accordingly:

| GPU runtime | dtype | Q3 batch / Br batch | First-run cell time \* | Steady-state RTF on 30 s clip |
|---|---|---|---|---|
| **B100 / B200** (Blackwell datacenter, 192 GB) | bf16 | 128 / 96 | ~70 s | ~250 x \*\* |
| **RTX Pro 6000** (Blackwell workstation sm_120, 96 GB) — Colab Pro+ | bf16 | 96 / 64 | ~70 s | ~250 x \*\* |
| **RTX 5090** (Blackwell sm_120, 32 GB) | bf16 | 48 / 32 | ~70 s | ~190 x |
| **H100** (Hopper sm_90, 80 GB) | bf16 | 32 / 48 | ~70 s | ~150 x |
| **A100 40 / 80 GB** (Ampere sm_80) — Colab Pro+ | bf16 | 24 / 24 | ~80 s | ~80 x |
| **L40 / L40S** (Ada sm_89, 48 GB) — Colab Pro+ | bf16 | 16 / 24 | ~85 s | ~70 x |
| **L4** (Ada sm_89, 24 GB) — Colab Pro | bf16 | 16 / 8-12 | ~85 s | ~30-50 x |
| **RTX 4090** (Ada sm_89, local) | bf16 | 16 / 8-12 | ~85 s | ~50 x |
| **RTX 3090 / A6000** (Ampere sm_86) | bf16 | 12 / 8 | ~95 s | ~30 x |
| **T4** (Turing sm_75, 16 GB) — free Colab | fp16 | 4 / 4 | ~110 s | ~10 x |
| **V100** (Volta sm_70, 16 GB) | fp16 | 4 / 4 | ~120 s | ~8 x |
| **CPU only** | fp32 | 1 / 1 | ~60 s + ~60 s inference | ~0.5 x (slow but works) |

\* First-run cell time = pip install + ~3 GB Breeze model download + CTranslate2 conversion.
Subsequent runs only do inference. Numbers in the last column extrapolate from the
RTX 5090 benchmark in [`docs/BENCHMARK.md`](https://github.com/thc1006/taiwan-asr-toolkit/blob/main/docs/BENCHMARK.md);
your mileage will vary by ±30 % depending on Colab disk speed.

\*\* RTF on a single 30 s clip is dominated by per-call overhead, so the larger-VRAM
Blackwell GPUs (RTX Pro 6000, B100/B200) only pull decisively ahead on **multi-file
batch jobs** (`asr-qwen3 *.mp3` with cross-file chunk pool batching) or long single
files where their bigger batch can stay full. For a 712-min mixed Taiwan-Mandarin
corpus on RTX 5090 we measured Breeze RTF 382x and Qwen3 RTF 354x; expect the same
hardware-stratified gap on Colab.

In [ ]:
# Cell 1 — install the toolkit
# This pulls torch from Colab's preinstalled wheel (no upgrade unless required),
# then installs the toolkit + its core deps.
import sys
!{sys.executable} -m pip install -q --upgrade pip
!{sys.executable} -m pip install -q --upgrade-strategy only-if-needed \
    'taiwan-asr-toolkit @ git+https://github.com/thc1006/taiwan-asr-toolkit.git@main'
# Optional extras (uncomment if you need them):
# !{sys.executable} -m pip install -q jiwer            # for asr-cer evaluation
# !{sys.executable} -m pip install -q pyannote.audio   # for asr-diarize speaker labels

In [ ]:
# Cell 2 — verify the runtime, GPU, and install
import importlib, sys, os

print('Python    :', sys.version.split()[0])

try:
    import torch
    print('PyTorch   :', torch.__version__, '|  CUDA build:', torch.version.cuda)
    if torch.cuda.is_available():
        cap = torch.cuda.get_device_capability(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
        cap_label = {(7,0):'Volta', (7,5):'Turing', (8,0):'Ampere', (8,6):'Ampere',
                     (8,9):'Ada', (9,0):'Hopper', (12,0):'Blackwell'}.get(tuple(cap), f'sm_{cap[0]}{cap[1]}')
        bf16 = 'yes' if torch.cuda.is_bf16_supported() else 'NO (will fall back to fp16)'
        print(f'GPU       : {torch.cuda.get_device_name(0)}  ({cap_label}, {vram:.0f} GB)')
        print(f'BF16 TC   : {bf16}')
    else:
        print('GPU       : (none — running on CPU; expect ~60 s on a 30 s clip)')
except ImportError:
    print('!! PyTorch not installed. Cell 1 must run first.')
    raise

import taiwan_asr
print('Toolkit   :', taiwan_asr.__version__)

# The CLI scripts may not be on PATH in Colab subshells; we use `python -m`.
import shutil
if shutil.which('asr-breeze'):
    print('CLI       : asr-breeze on PATH')
else:
    print('CLI       : asr-breeze not on PATH (will use python -m taiwan_asr.breeze)')

In [ ]:
# Cell 3 — fetch the bundled 30 s sample + the default NTU glossary
import urllib.request
from pathlib import Path

BASE = 'https://raw.githubusercontent.com/thc1006/taiwan-asr-toolkit/main'
downloads = [
    (f'{BASE}/tests/fixtures/clip_30s.wav', 'clip_30s.wav'),
    (f'{BASE}/glossary.txt',                'glossary.txt'),
]
for src, dst in downloads:
    p = Path(dst)
    if not p.is_file():
        urllib.request.urlretrieve(src, dst)
        print(f'downloaded  {dst}  ({p.stat().st_size/1024:.0f} KB)')
    else:
        print(f'cached      {dst}')

In [ ]:
# Cell 4 — run Breeze-ASR-25 with hot-word injection
# The first run downloads the Breeze-ASR-25 weights (~3 GB) and converts them
# to CTranslate2 format (~30 s). Subsequent runs are pure inference.
import sys, time
t0 = time.time()
!{sys.executable} -m taiwan_asr.breeze clip_30s.wav --glossary-file glossary.txt
print(f'\nWall time: {time.time()-t0:.1f} s')

In [ ]:
# Cell 5 — show the transcript (Traditional Chinese with timestamps)
from pathlib import Path
out_txt = Path('transcripts/breeze/clip_30s_breeze.txt')
out_srt = Path('transcripts/breeze/clip_30s_breeze.srt')
out_json = Path('transcripts/breeze/clip_30s_breeze.json')

if out_txt.is_file():
    print('=== TXT ===')
    print(out_txt.read_text(encoding='utf-8'))
    print('\n=== SRT (first 8 lines) ===')
    print('\n'.join(out_srt.read_text(encoding='utf-8').splitlines()[:8]))
    import json
    segs = json.loads(out_json.read_text(encoding='utf-8'))
    print(f'\n=== JSON: {len(segs)} segment(s), word-level timestamps available ===')
else:
    print(f'!! expected output {out_txt} not found — check Cell 4 stderr above')

## What you just saw

- Output is **Traditional Chinese (Taiwan)** thanks to OpenCC `s2twp` post-processing baked into the pipeline (簡體 `软件` would have become `軟體`).
- The transcript should mention **`研三`** (NTU graduate dorm) — the hot-word injection biased Whisper toward it. Without `--glossary-file`, Breeze tends to mistranscribe this as `圓三` (a non-existent dorm). See [`examples/compare_alternatives.md`](https://github.com/thc1006/taiwan-asr-toolkit/blob/main/examples/compare_alternatives.md) for the side-by-side proof.
- TXT, SRT, and JSON were all written; the JSON has segment-level + word-level timestamps you can feed downstream tools.

In [ ]:
# Cell 6 (optional) — also run Qwen3-ASR for cross-model comparison
# Qwen3-ASR-1.7B is a different architecture (multi-lingual transformer encoder-decoder
# with optional ForcedAligner-0.6B for word-level timestamps).  Useful as a sanity check.
# First run downloads ~5 GB of Qwen3-ASR + Aligner weights.
import sys
!{sys.executable} -m taiwan_asr.qwen3 clip_30s.wav --no-aligner   # --no-aligner saves ~25 % time
from pathlib import Path
qpath = Path('transcripts/qwen3/clip_30s_qwen3.txt')
if qpath.is_file():
    print('=== Qwen3 transcript ===')
    print(qpath.read_text(encoding='utf-8'))

## Try your own audio

```python
from google.colab import files
uploaded = files.upload()                         # opens a file picker
name = next(iter(uploaded))                       # first uploaded file name
!python -m taiwan_asr.breeze "{name}" --glossary-file glossary.txt
```

Supported formats: anything ffmpeg can read (mp3, m4a, wav, mp4, flac, ogg…).
On a 1-hour audio file, expect roughly 30 s on an A100, 5 minutes on a T4, or
3 minutes on the free-tier CPU runtime.

## Next steps

- **LLM context polish** (fix homophones using a Qwen3-8B reasoning pass while protecting NTU proper nouns):
  ```bash
  python -m taiwan_asr.polish transcripts/breeze/clip_30s_breeze.json --glossary-file glossary.txt
  ```
- **Speaker diarization** (requires HuggingFace license accept on `pyannote/speaker-diarization-3.1`, `pyannote/speaker-diarization-community-1`, `pyannote/segmentation-3.0`):
  ```bash
  python -m taiwan_asr.diarize transcripts/breeze/clip_30s_breeze.json clip_30s.wav
  ```
- **CER evaluation** against your own ground truth:
  ```bash
  python -m taiwan_asr.cer_eval --ref my_gt.txt --hyp transcripts/breeze/clip_30s_breeze.json --clip-end 30
  ```
- **Benchmark suite** on a folder of audio files — see [`docs/BENCHMARK.md`](https://github.com/thc1006/taiwan-asr-toolkit/blob/main/docs/BENCHMARK.md).
- **Custom glossaries**: edit `glossary.txt` (one term per line, `#` for comments).
  Hot-word injection is the main mechanism for fixing domain-specific proper nouns at the source.

## License & credits

Toolkit: MIT.
Models retain their own licenses (Apache 2.0 for Qwen and Breeze, gated for pyannote — please cite the underlying authors).